In [12]:
import pandas as pd
import numpy as np

# ── 1. LOAD ─────────────────────────────────────────────
df = pd.read_csv(r"D:\assesment project\Campaign_Raw.csv", dtype=str)
print(f"Raw rows: {len(df)}")

# ── 2. CLEAN COLUMN NAMES (CRITICAL) ────────────────────
df.columns = (
    df.columns
    .str.strip()
    .str.replace('\n', '', regex=True)
)

# ── 3. CLEAN VALUES ─────────────────────────────────────
df = df.apply(lambda col: col.map(lambda x: x.strip() if isinstance(x, str) else x))

# ── 4. HANDLE MISSING VALUES ────────────────────────────
df.replace(["NAN", "nan", "Nan", ""], np.nan, inplace=True)

# ── 5. CLEAN STRING COLUMNS ─────────────────────────────
df["Data Source name"] = df["Data Source name"].str.title()
df["Campaign Effective Status"] = df["Campaign Effective Status"].str.upper()
df["Country Funnel"] = df["Country Funnel"].str.title()
df["Geo Location Segment"] = df["Geo Location Segment"].str.title()
df["Ad Set Name"] = df["Ad Set Name"].str.strip()
df["Campaign Name"] = df["Campaign Name"].str.strip()
df["Ad Name"] = df["Ad Name"].str.strip()

# ── 6. FIX DATE ─────────────────────────────────────────
df["Date"] = pd.to_datetime(df["Date"], format="%d-%m-%Y", errors="coerce")
print(f"Invalid dates: {df['Date'].isna().sum()}")

# ── 7. CONVERT NUMERIC COLUMNS ──────────────────────────
numeric_cols = [
    "FB Spent Funnel (INR)", "Amount Spent (INR)", "Clicks (all)",
    "Impressions", "Page Likes", "Landing Page Views", "Link Clicks",
    "Adds to Cart", "Checkouts Initiated", "Adds of Payment Info",
    "Purchases", "Purchases Conversion Value (INR)", "Website Contacts",
    "Messaging Conversations Started",
    "Adds to Cart Conversion Value (INR)",
    "Checkouts Initiated Conversion Value (INR)",
    "Adds of Payment Info Conversion Value (INR)"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ── 8. FIX NEGATIVE VALUES ──────────────────────────────
fix_zero_cols = ["Amount Spent (INR)", "Clicks (all)", "Impressions",
                 "Link Clicks", "Purchases"]

for col in fix_zero_cols:
    if col in df.columns:
        neg_count = (df[col] < 0).sum()
        print(f"Negative values in '{col}': {neg_count} → set to 0")
        df[col] = df[col].clip(lower=0)

# ── 9. CALCULATED METRICS ───────────────────────────────
df["CTR_calculated"] = np.where(
    df["Impressions"] > 0,
    (df["Clicks (all)"] / df["Impressions"]) * 100,
    0
)

df["CPM_calculated"] = np.where(
    df["Impressions"] > 0,
    (df["Amount Spent (INR)"] / df["Impressions"]) * 1000,
    0
)

df["CPC_calculated"] = np.where(
    df["Clicks (all)"] > 0,
    df["Amount Spent (INR)"] / df["Clicks (all)"],
    0
)

df["ROI_calculated"] = np.where(
    df["Amount Spent (INR)"] > 0,
    ((df["Purchases Conversion Value (INR)"] - df["Amount Spent (INR)"])
     / df["Amount Spent (INR)"]) * 100,
    0
)

df["ROAS_calculated"] = np.where(
    df["Amount Spent (INR)"] > 0,
    df["Purchases Conversion Value (INR)"] / df["Amount Spent (INR)"],
    0
)

# ── 10. REMOVE DUPLICATES ───────────────────────────────
subset_cols = ["Date", "Campaign Name", "Ad Name",
               "Amount Spent (INR)", "Clicks (all)"]

before = len(df)
df.drop_duplicates(subset=subset_cols, keep="first", inplace=True)
print(f"Duplicates removed: {before - len(df)}")

# ── 11. DROP UNUSED COLUMN ──────────────────────────────
df.drop(columns=["Row Count"], errors="ignore", inplace=True)

# ── 12. RENAME FOR SQL ──────────────────────────────────
df.rename(columns={
    "Data Source name": "brand_name",
    "Date": "date",
    "Campaign Name": "campaign_name",
    "Campaign Effective Status": "status",
    "Ad Set Name": "ad_set_name",
    "Ad Name": "ad_name",
    "Country Funnel": "country",
    "Geo Location Segment": "region",
    "Amount Spent (INR)": "spend_inr",
    "Clicks (all)": "clicks",
    "Impressions": "impressions",
    "Page Likes": "page_likes",
    "Landing Page Views": "landing_page_views",
    "Link Clicks": "link_clicks",
    "Adds to Cart": "adds_to_cart",
    "Checkouts Initiated": "checkouts_initiated",
    "Purchases": "purchases",
    "Purchases Conversion Value (INR)": "revenue_inr",
    "Website Contacts": "website_contacts"
}, inplace=True)

# ── 13. SAVE ────────────────────────────────────────────
df.to_csv(r"D:\assesment project\cleaned_Campaign_Raw.csv", index=False)

print(f"Final rows: {len(df)}")
print("✅ Cleaned campaign file saved successfully!")

Raw rows: 10028
Invalid dates: 673
Negative values in 'Amount Spent (INR)': 425 → set to 0
Negative values in 'Clicks (all)': 373 → set to 0
Negative values in 'Impressions': 429 → set to 0
Negative values in 'Link Clicks': 362 → set to 0
Negative values in 'Purchases': 34 → set to 0
Duplicates removed: 1304
Final rows: 8724
✅ Cleaned campaign file saved successfully!


In [13]:
import pandas as pd
import numpy as np

# ── 1. LOAD ─────────────────────────────────────────────
df = pd.read_csv(r"D:\assesment project\Raw_Shopify_Sales.csv", dtype=str)
print(f"Raw rows: {len(df)}")

# ── 2. CLEAN COLUMN NAMES ───────────────────────────────
df.columns = df.columns.str.strip().str.replace('\n', '', regex=True)

# ── 3. CLEAN VALUES ─────────────────────────────────────
df = df.apply(lambda col: col.map(lambda x: x.strip() if isinstance(x, str) else x))

# ── 4. HANDLE NULLS ─────────────────────────────────────
df.replace(["NAN", "nan", "Nan", ""], np.nan, inplace=True)

# ── 5. CLEAN STRING COLUMNS ─────────────────────────────
df["Data Source name"] = df["Data Source name"].str.title()
df["Sales Channel"] = df["Sales Channel"].str.title()
df["Country Funnel"] = df["Country Funnel"].str.title()
df["Geo Location Segment"] = df["Geo Location Segment"].str.title()
df["Billing Country"] = df["Billing Country"].str.title()
df["Billing City"] = df["Billing City"].str.title()

# ── 6. FIX DATE COLUMNS ─────────────────────────────────
df["Date"] = pd.to_datetime(df["Date"], format="%d-%m-%Y", errors="coerce")
df["Order Created At"] = pd.to_datetime(df["Order Created At"], errors="coerce")
df["Order Updated At"] = pd.to_datetime(df["Order Updated At"], errors="coerce")
df["Transaction Timestamp"] = pd.to_datetime(df["Transaction Timestamp"], errors="coerce")

print(f"Invalid Date rows: {df['Date'].isna().sum()}")

# ── 7. CONVERT NUMERIC COLUMNS ──────────────────────────
numeric_cols = [
    "Gross Sales (INR)", "Net Sales (INR)", "Total Sales (INR)",
    "Orders", "Returns (INR)", "Return Rate",
    "Items Sold", "Items Returned",
    "Average Order Value (INR)", "New Customer Orders",
    "Returning Customer Orders", "Average Items Per Order",
    "Discounts (INR)"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ── 8. FIX NEGATIVE VALUES ──────────────────────────────
for col in numeric_cols:
    df[col] = df[col].clip(lower=0)

# ── 9. REMOVE DUPLICATES ───────────────────────────────
before = len(df)
df.drop_duplicates(subset=["Order ID"], keep="first", inplace=True)
print(f"Duplicates removed: {before - len(df)}")

# ── 10. DROP UNUSED COLUMN ─────────────────────────────
df.drop(columns=["Row Count"], errors="ignore", inplace=True)

# ── 11. RENAME FOR SQL ─────────────────────────────────
df.rename(columns={
    "Data Source name": "brand_name",
    "Date": "date",
    "Currency": "currency",
    "Sales Channel": "sales_channel",
    "Transaction Timestamp": "transaction_timestamp",
    "Order Created At": "order_created_at",
    "Order Updated At": "order_updated_at",
    "Order ID": "order_id",
    "Order Name": "order_name",
    "Country Funnel": "country",
    "Geo Location Segment": "region",
    "Billing Country": "billing_country",
    "Billing Province": "billing_province",
    "Billing City": "billing_city",
    "Order Tags": "order_tags",
    "Product ID": "product_id",
    "Product Title": "product_title",
    "Product Tags": "product_tags",
    "Product Type": "product_type",
    "Variant Title": "variant_title",
    "Gross Sales (INR)": "gross_sales",
    "Net Sales (INR)": "net_sales",
    "Total Sales (INR)": "total_sales",
    "Orders": "orders",
    "Returns (INR)": "returns_inr",
    "Return Rate": "return_rate",
    "Items Sold": "items_sold",
    "Items Returned": "items_returned",
    "Average Order Value (INR)": "avg_order_value",
    "New Customer Orders": "new_customer_orders",
    "Returning Customer Orders": "returning_customer_orders",
    "Average Items Per Order": "avg_items_per_order",
    "Discounts (INR)": "discounts_inr",
    "SKU": "sku",
    "Customer Sale Type": "customer_sale_type",
    "Customer ID": "customer_id",
    "Shipping Country": "shipping_country"
}, inplace=True)

# ── 12. SAVE ───────────────────────────────────────────
df.to_csv(r"D:\assesment project\cleaned_Shopify_Sales.csv", index=False)

print(f"Final rows: {len(df)}")
print("✅ Cleaned Shopify file saved successfully!")

Raw rows: 5680
Invalid Date rows: 482
Duplicates removed: 3507
Final rows: 2173
✅ Cleaned Shopify file saved successfully!
